# UVA + CoT/LLM wrapper → LIBERO-10 평가 (Colab)

동결된 UVA(`libero10.ckpt`) 위에 **추론 시점 CoT 오케스트레이션**을 씌워 LIBERO-10을 평가합니다.

| 모드 | 설명 |
|------|------|
| `no_cot=True` | 논문 baseline (`eval_sim.py`와 동일) |
| `planner="rule"` | 규칙 기반 CoT → `language_goal` 재작성 (API 불필요) |
| `planner="llm"` | OpenAI vision CoT (`gpt-4o-mini` 등, **API 키 필요**) |

**비교 목표**: baseline vs rule-CoT vs LLM-CoT 의 `test_mean_score`.

> 이미 `colab_libero10_eval.ipynb`로 4b까지 설치했다면 **셀 0 → 8(OpenAI 키) → 9(함수) → 10~** 만 실행해도 됩니다.

> **런타임**: GPU (L4/T4/A100). `MUJOCO_GL=egl` 필수. LLM 모드는 Colab Secrets에 `OPENAI_API_KEY` 등록.

## 0. Python 3.10 설치

★ 처음 한 번만 실행. 자동 재시작 후 Python 버전 확인하고 다음 셀부터 진행.

In [ ]:
import sys
print('Python:', sys.version)
# Python 3.10 설치 불필요 — mujoco-py 의존성이 제거되어 3.12에서 직접 실행 가능합니다.


## 1. 런타임 체크 + headless GL

In [ ]:
import os, subprocess
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MKL_THREADING_LAYER"] = "GNU"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. Google Drive (선택)

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_ROOT = "/content/drive/MyDrive/uva_libero"
    os.makedirs(PERSIST_ROOT, exist_ok=True)
    print("Drive:", PERSIST_ROOT)
else:
    PERSIST_ROOT = None
    print("Drive 미사용")

## 3. apt 패키지 (EGL / MuJoCo)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libegl1 libegl1-mesa libgles2-mesa libgl1-mesa-glx \
    libosmesa6 libosmesa6-dev libglfw3 libglew-dev patchelf ffmpeg > /dev/null
!apt-get install -y -qq python3-dev build-essential pkg-config \
    libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev > /dev/null
# mujoco-py 불필요 — mujoco==3.x 직접 사용 (_mujoco_py_shim 처리됨)


## 4. 레포 클론

In [ ]:
#import os

UVA_REPO_URL    = "https://github.com/zser99/Unified-Video-Action-with-LLM.git"
UVA_REPO_BRANCH = "main"

%cd /content
if not os.path.isdir("/content/unified_video_action"):
    !git clone --depth 1 -b {UVA_REPO_BRANCH} {UVA_REPO_URL} unified_video_action
if not os.path.isdir("/content/LIBERO"):
    !git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
!ls /content/unified_video_action/unified_video_action/cot 2>/dev/null || echo "WARNING: cot/ folder missing — fork URL 확인"

## 5. 패키지 설치

In [ ]:
import sys, subprocess, os

def _pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("\n[pip STDERR]\n" + r.stderr[-4000:])
    if check and r.returncode != 0:
        raise subprocess.CalledProcessError(r.returncode, cmd, r.stdout, r.stderr)
    return r

print("Python:", sys.version)

_pip("install", "-q", "gym==0.26.2")
import gym; print("gym OK:", gym.__version__)

# numpy는 Colab 기본값(2.x) 그대로 사용 — 다운그레이드하면 커널 재시작 필요
_pip("install", "-q",
     "numpy>=2.0.0", "scipy>=1.13.0", "numba>=0.61.0",
     "zarr>=2.18.0", "numcodecs>=0.12.1", "h5py", "dill==0.3.7")

_pip("install", "-q",
     "hydra-core==1.3.2", "omegaconf==2.3.0", "antlr4-python3-runtime==4.9.3", "einops==0.6.1",
     "diffusers==0.18.2", "transformers==4.40.0", "huggingface_hub==0.25.2",
     "timm==0.9.7", "accelerate==1.0.1")

_pip("install", "-q",
     "imagecodecs==2024.12.30", "kornia==0.8.0", "opencv-python==4.11.0.86",
     "wandb==0.18.3", "gdown==5.2.0", "click", "pandas", "openai>=1.40.0")

_pip("install", "-q", "av", check=False)
_pip("install", "-q", "mujoco==3.1.6", "robosuite==1.4.1")
_pip("install", "-q", "robomimic==0.3.0", check=False)
_pip("install", "-q", "-e", "/content/LIBERO", "--no-deps")
_pip("install", "-q", "-e", "/content/unified_video_action", "--no-deps")
print("\n패키지 설치 완료")


## 6. 체크포인트 (~3 GB)

In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")
%cd /content/unified_video_action

if PERSIST_ROOT:
    _ckpt_dir = os.path.join(PERSIST_ROOT, "checkpoints")
    os.makedirs(_ckpt_dir, exist_ok=True)
    if not os.path.exists("checkpoints"):
        os.symlink(_ckpt_dir, "checkpoints")
    CKPT = os.path.join(_ckpt_dir, "libero10.ckpt")
else:
    os.makedirs("checkpoints", exist_ok=True)
    CKPT = "checkpoints/libero10.ckpt"

if not os.path.isfile(CKPT):
    !gdown 11c2VrmaRp48yw__5A5xpcu8EPzkexHSi -O {CKPT}
!ls -lh checkpoints

## 7. LIBERO-10 hdf5

In [ ]:
import glob, os
%cd /content/unified_video_action

if PERSIST_ROOT:
    _data_dir = os.path.join(PERSIST_ROOT, "libero_10")
    os.makedirs(_data_dir, exist_ok=True)
    os.makedirs("data", exist_ok=True)
    if not os.path.exists("data/libero_10"):
        os.symlink(_data_dir, "data/libero_10")
else:
    _data_dir = "data/libero_10"
    os.makedirs(_data_dir, exist_ok=True)

if len(glob.glob(f"{_data_dir}/*.hdf5")) < 10:
    !gdown 1_6Kc7e-s30MblbX8YjpxSofe9ZRPk3xv -O /tmp/libero_10_raw.zip
    !unzip -oq /tmp/libero_10_raw.zip -d /tmp/libero_unzip/
    !find /tmp/libero_unzip -name "*.hdf5" -exec cp {{}} {_data_dir}/ \;
    !rm -rf /tmp/libero_10_raw.zip /tmp/libero_unzip

hdf5 = sorted(glob.glob(f"{_data_dir}/*.hdf5"))
print(len(hdf5), "files"); assert len(hdf5) == 10

## 8. 초기화 ← 재시작할 때마다 이 셀 하나만 실행

In [10]:
"""재시작 후 초기화 — 이 셀 하나로 환경·shim 전부 처리"""
import os, sys, subprocess, types, importlib
from pathlib import Path

# ── 1. 환경변수 ──────────────────────────────────────────────
os.environ["MUJOCO_GL"]          = "egl"
os.environ["PYOPENGL_PLATFORM"]  = "egl"
os.environ["MKL_THREADING_LAYER"] = "GNU"
if not os.path.isdir("/content/unified_video_action"):
    raise RuntimeError("새 런타임: 셀 3(apt)→4(clone)→5(pip)→6(ckpt)→7(hdf5) 먼저 실행하세요")
os.chdir("/content/unified_video_action")

# ── 2. sys.path ───────────────────────────────────────────────
for p in ("/content/LIBERO", "/content/unified_video_action"):
    if p not in sys.path:
        sys.path.insert(0, p)

# ── 3. 필수 패키지 ───────────────────────────────────────────
for pkg, pip_name in [("bddl", "bddl==1.0.1"), ("easydict", "easydict")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)

# ── 4. mujoco_py shim ─────────────────────────────────────────
try:
    import mujoco_py  # noqa: F401
    print("mujoco_py: real")
except ImportError:
    import mujoco
    sys.modules["mujoco_py"] = mujoco
    print("mujoco_py: shim OK (mujoco", mujoco.__version__, ")")

# ── 4b. jax.random.KeyArray compat (accelerate→flax→jax 연쇄 대응) ──
try:
    import jax, jax.random
    if not hasattr(jax.random, "KeyArray"):
        jax.random.KeyArray = jax.Array
        print("jax.random.KeyArray: compat shim OK")
    else:
        print("jax.random.KeyArray: native OK")
except (ImportError, AttributeError):
    pass  # JAX 없으면 무시

# ── 4c. hydra Python 3.12 호환 패치 후 초기화 ──────────────
import subprocess as _sp_h, re as _re_h, pathlib as _pl_h
_loc_m = _re_h.search(
    r'Location:\s*(.+)',
    _sp_h.run([sys.executable, '-m', 'pip', 'show', 'hydra-core'],
              capture_output=True, text=True).stdout
)
if _loc_m:
    _hconf = _pl_h.Path(_loc_m.group(1).strip()) / 'hydra' / 'conf' / '__init__.py'
    if _hconf.exists():
        _htxt = _hconf.read_text('utf-8')
        if 'OverrideDirname = OverrideDirname()' in _htxt and 'default_factory' not in _htxt:
            _htxt = _htxt.replace(
                'OverrideDirname = OverrideDirname()',
                'OverrideDirname = field(default_factory=OverrideDirname)'
            )
            _hconf.write_text(_htxt, 'utf-8')
            print('hydra: patched mutable default for Python 3.12')
del _sp_h, _re_h, _pl_h
import hydra
print("hydra:", hydra.__version__)

# ── 5. pytorch3d shim ─────────────────────────────────────────
import torch, numpy as np
from scipy.spatial.transform import Rotation as R

def _to_numpy(x):
    if isinstance(x, torch.Tensor): return x.detach().cpu().numpy(), x
    return np.asarray(x), None

def _to_like(arr, like):
    if like is None: return arr
    return torch.from_numpy(arr).to(device=like.device, dtype=like.dtype)

def axis_angle_to_matrix(x):
    xn, like = _to_numpy(x)
    return _to_like(R.from_rotvec(xn.reshape(-1,3)).as_matrix().reshape(*xn.shape[:-1],3,3), like)

def matrix_to_axis_angle(x):
    xn, like = _to_numpy(x)
    return _to_like(R.from_matrix(xn.reshape(-1,3,3)).as_rotvec().reshape(*xn.shape[:-2],3), like)

def matrix_to_rotation_6d(x):
    if isinstance(x, torch.Tensor): return x[...,:2,:].reshape(*x.shape[:-2],6)
    xn,_ = _to_numpy(x); return xn[...,:2,:].reshape(*xn.shape[:-2],6)

def rotation_6d_to_matrix(x):
    if isinstance(x, torch.Tensor):
        a1,a2 = x[...,0:3], x[...,3:6]
        b1 = torch.nn.functional.normalize(a1, dim=-1)
        b2 = torch.nn.functional.normalize(a2-(b1*a2).sum(-1,keepdim=True)*b1, dim=-1)
        return torch.stack((b1, b2, torch.cross(b1,b2,dim=-1)), dim=-2)
    xn = np.asarray(x); a1,a2 = xn[...,0:3], xn[...,3:6]
    b1 = a1/(np.linalg.norm(a1,axis=-1,keepdims=True)+1e-8)
    b2 = a2-(b1*a2).sum(axis=-1,keepdims=True)*b1
    b2 = b2/(np.linalg.norm(b2,axis=-1,keepdims=True)+1e-8)
    return np.stack((b1, b2, np.cross(b1,b2,axis=-1)), axis=-2)

_pt = types.ModuleType("pytorch3d.transforms")
for _fn in [axis_angle_to_matrix, matrix_to_axis_angle, matrix_to_rotation_6d, rotation_6d_to_matrix]:
    setattr(_pt, _fn.__name__, _fn)
_p3d = types.ModuleType("pytorch3d"); _p3d.transforms = _pt
sys.modules.update({"pytorch3d": _p3d, "pytorch3d.transforms": _pt})
print("pytorch3d: shim OK")

# ── 6. LiberoImageRunner ──────────────────────────────────────
import unified_video_action.env_runner.libero_image_runner as _lir
importlib.reload(_lir)
from unified_video_action.env_runner.libero_image_runner import LiberoImageRunner
print("LiberoImageRunner: OK")


# ── 6b. Colab worker 크래시 패치 (git pull 없어도 동작) ────────────────────
if os.path.isdir('/content'):
    import subprocess as _sp
    _pull = _sp.run(['git', 'pull', 'origin', 'main'],
                    cwd='/content/unified_video_action',
                    capture_output=True, text=True)
    print('git pull:', (_pull.stdout + _pull.stderr).strip()[:120])

    # Clear ALL UVA module cache so the reloaded file is used
    _stale = [k for k in list(sys.modules.keys()) if 'unified_video_action' in k]
    for _k in _stale:
        del sys.modules[_k]
    print(f'Cleared {len(_stale)} cached modules')

    import unified_video_action.env_runner.libero_image_runner as _lir
    importlib.reload(_lir)
    from unified_video_action.env_runner.libero_image_runner import LiberoImageRunner

    # SyncVectorEnv runs in-process (no fork) so EGL GPU rendering is safe
    from unified_video_action.gym_util.sync_vector_env import SyncVectorEnv as _SV

    class _ColabSyncEnv(_SV):
        def __init__(self, env_fns, dummy_env_fn=None, shared_memory=False,
                     copy=True, context=None, daemon=True, worker=None):
            super().__init__(env_fns, copy=copy)
    _lir.AsyncVectorEnv = _ColabSyncEnv

    print('Colab patch OK: EGL GPU rendering + in-process SyncVectorEnv')

# ── 7. CoT 모듈 확인 ──────────────────────────────────────────
from unified_video_action.cot.factory import create_planner
from unified_video_action.policy.cot_orchestrated_policy import CoTOrchestratedPolicy
print("CoT modules: OK")
print("\u2501" * 45)
print("초기화 완료 → 셀 9 (OpenAI 키) → 셀 10 (함수 정의) → 셀 11 (실험)")


mujoco_py: real
jax.random.KeyArray: native OK
hydra: 1.3.2
pytorch3d: shim OK
LiberoImageRunner: OK


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


git pull: Updating caedd18..736559d
Fast-forward
 unified_video_action/cot/factory.py | 1 +
 1 file changed, 1 insertion(+)
From h
Cleared 47 cached modules
Colab patch OK: EGL GPU rendering + in-process SyncVectorEnv
CoT modules: OK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
초기화 완료 → 셀 9 (OpenAI 키) → 셀 10 (함수 정의) → 셀 11 (실험)


## 8b. EGL 진단 (셀 8 후 바로 실행 — hang 시 osmesa로 전환)

In [ ]:
import os, time, threading

os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

_result = {}

def _egl_test():
    try:
        import mujoco
        m = mujoco.MjModel.from_xml_string("<mujoco></mujoco>")
        d = mujoco.MjData(m)
        renderer = mujoco.Renderer(m, height=64, width=64)
        renderer.update_scene(d)
        img = renderer.render()
        _result["ok"] = img.shape
    except Exception as e:
        _result["error"] = str(e)

t = threading.Thread(target=_egl_test, daemon=True)
t0 = time.time()
t.start()
t.join(timeout=15)

if t.is_alive():
    print("❌ EGL HANG — MuJoCo renderer did not return in 15s")
    print()
    print("→ osmesa로 전환하려면 셀 1과 셀 8 상단의 환경변수를 아래로 교체하세요:")
    print('   os.environ["MUJOCO_GL"] = "osmesa"')
    print('   os.environ["PYOPENGL_PLATFORM"] = "osmesa"  # 또는 이 줄 삭제')
    print()
    print("  그리고 셀 8을 다시 실행 후 셀 10 → 셀 11 순서로 진행.")
    # Apply osmesa fallback immediately
    os.environ["MUJOCO_GL"] = "osmesa"
    try:
        del os.environ["PYOPENGL_PLATFORM"]
    except KeyError:
        pass
    print("⚠️  이 세션에서는 osmesa로 자동 전환됨. 셀 8을 반드시 다시 실행하세요.")
elif "error" in _result:
    print(f"❌ EGL ERROR ({time.time()-t0:.1f}s): {_result['error']}")
    print("→ osmesa 전환 필요. 위 메시지 참고.")
    os.environ["MUJOCO_GL"] = "osmesa"
    try:
        del os.environ["PYOPENGL_PLATFORM"]
    except KeyError:
        pass
else:
    print(f"✅ EGL OK ({time.time()-t0:.1f}s): rendered {_result['ok']}")
    print("GPU rendering 사용 가능 — 셀 9 → 셀 10 → 셀 11 진행")

## 9. OpenAI API 키 (LLM CoT용)

In [14]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OPENAI_API_KEY: loaded from Colab Secrets (", len(os.environ["OPENAI_API_KEY"]), "chars)")
except Exception as e:
    print("Secrets 없음:", e)
    print("수동 설정: os.environ['OPENAI_API_KEY'] = 'sk-...'")
# os.environ["OPENAI_API_KEY"] = "sk-..."  # 필요 시 주석 해제

OPENAI_API_KEY: loaded from Colab Secrets ( 164 chars)


## 10. 평가 함수 정의

In [15]:
# pytorch3d shim guard -- works even if cell 8 was skipped
import sys as _sys, types as _types
if 'pytorch3d' not in _sys.modules:
    import torch as _t2, numpy as _np2
    from scipy.spatial.transform import Rotation as _R2
    def _tn(x): return (x.detach().cpu().numpy(), x) if hasattr(x,'detach') else (_np2.asarray(x), None)
    def _tl(a,l): return _t2.from_numpy(a).to(device=l.device,dtype=l.dtype) if l is not None else a
    def axis_angle_to_matrix(x):
        n,l=_tn(x); return _tl(_R2.from_rotvec(n.reshape(-1,3)).as_matrix().reshape(*n.shape[:-1],3,3),l)
    def matrix_to_axis_angle(x):
        n,l=_tn(x); return _tl(_R2.from_matrix(n.reshape(-1,3,3)).as_rotvec().reshape(*n.shape[:-2],3),l)
    def matrix_to_rotation_6d(x):
        if hasattr(x,'detach'): return x[...,:2,:].reshape(*x.shape[:-2],6)
        n=_np2.asarray(x); return n[...,:2,:].reshape(*n.shape[:-2],6)
    def rotation_6d_to_matrix(x):
        if hasattr(x,'detach'):
            a1,a2=x[...,0:3],x[...,3:6]; b1=_t2.nn.functional.normalize(a1,dim=-1)
            b2=_t2.nn.functional.normalize(a2-(b1*a2).sum(-1,keepdim=True)*b1,dim=-1)
            return _t2.stack((b1,b2,_t2.cross(b1,b2,dim=-1)),dim=-2)
        n=_np2.asarray(x); a1,a2=n[...,0:3],n[...,3:6]
        b1=a1/(_np2.linalg.norm(a1,axis=-1,keepdims=True)+1e-8)
        b2=a2-(b1*a2).sum(axis=-1,keepdims=True)*b1; b2/=(_np2.linalg.norm(b2,axis=-1,keepdims=True)+1e-8)
        return _np2.stack((b1,b2,_np2.cross(b1,b2,axis=-1)),axis=-2)
    _pt2=_types.ModuleType('pytorch3d.transforms')
    for _fn2 in [axis_angle_to_matrix,matrix_to_axis_angle,matrix_to_rotation_6d,rotation_6d_to_matrix]:
        setattr(_pt2,_fn2.__name__,_fn2)
    _p3d2=_types.ModuleType('pytorch3d'); _p3d2.transforms=_pt2
    _sys.modules.update({'pytorch3d':_p3d2,'pytorch3d.transforms':_pt2})
    print('pytorch3d: shim auto-applied (cell 8 was not run)')

import os, sys, json, glob, random, pathlib, dill, hydra, torch, numpy as np, wandb, time
from omegaconf import open_dict

%cd /content/unified_video_action
sys.path.insert(0, "/content/unified_video_action")

from unified_video_action.workspace.base_workspace import BaseWorkspace
from unified_video_action.env_runner.base_image_runner import BaseImageRunner
from unified_video_action.cot.factory import create_planner
from unified_video_action.policy.cot_orchestrated_policy import CoTOrchestratedPolicy


def _get_libero_hdf5_files(cfg, task_subset):
    hdf5_files = sorted(glob.glob(cfg.task.dataset.dataset_path + "/*hdf5"))
    print(f"Found {len(hdf5_files)} HDF5 files in {cfg.task.dataset.dataset_path}:")
    for f in hdf5_files:
        print("  ", os.path.basename(f))
    if task_subset:
        # case-insensitive substring match
        hdf5_files = [f for f in hdf5_files
                      if any(s.lower() in os.path.basename(f).lower() for s in task_subset)]
        assert hdf5_files, f"task_subset {task_subset} matched nothing in above list"
    print(f"Running {len(hdf5_files)} task(s)")
    return hdf5_files


def run_libero10_cot_eval(
    checkpoint="checkpoints/libero10.ckpt",
    output_dir="outputs/libero10_cot",
    device="cuda:0",
    n_test=1,
    n_train=0,
    n_envs=1,
    n_test_vis=0,
    max_steps=None,
    task_subset=None,
  # --- CoT ---
    no_cot=False,
    planner="llm",
    llm_model="gpt-4o-mini",
    replan_every=8,
    num_candidates=1,
    candidate_strategy="first",
    verbose_cot=False,
    no_llm_fallback=False,
):
    t_start = time.time()
    pathlib.Path(output_dir).mkdir(parents=True, exist_ok=True)
    if device.startswith("cuda") and not torch.cuda.is_available():
        print("WARNING: no CUDA, using cpu"); device = "cpu"

    print(f"[1] Loading checkpoint: {checkpoint}")
    payload = torch.load(open(checkpoint, "rb"), pickle_module=dill, map_location="cpu")
    print(f"    done ({time.time()-t_start:.1f}s)")

    cfg = payload["cfg"]
    print(f"    cfg.task.name: {cfg.task.name}")
    seed = cfg.training.seed
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    with open_dict(cfg):
        cfg.output_dir = output_dir
        cfg.task.env_runner.n_test = int(n_test)
        cfg.task.env_runner.n_train = int(n_train)
        cfg.task.env_runner.n_envs = int(n_envs)
        cfg.task.env_runner.n_test_vis = min(n_test_vis, n_test)
        cfg.task.env_runner.n_train_vis = 0
        if max_steps is not None:
            cfg.task.env_runner.max_steps = int(max_steps)

    # ── [2] Build policy directly — skip workspace deepcopy + optimizer ──────
    print(f"[2] Building policy (no workspace deepcopy / no optimizer)...")
    language_emb_model = cfg.task.dataset.language_emb_model
    if language_emb_model == "clip":
        print(f"    [2a] Pre-caching CLIP from HuggingFace...")
        from transformers import AutoTokenizer, CLIPModel
        _tok = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")
        _clp = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        del _tok, _clp
        print(f"    CLIP cached ({time.time()-t_start:.1f}s)")

    print(f"    [2b] Instantiating policy...")
    inner = hydra.utils.instantiate(
        cfg.model.policy,
        task_name=cfg.task.name,
        task_modes=cfg.task.task_modes,
        normalizer_type=cfg.task.dataset.normalizer_type,
        language_emb_model=language_emb_model,
    )
    print(f"    policy built ({time.time()-t_start:.1f}s)")

    print(f"    [2c] Loading ema_model state dict...")
    ema_key = "ema_model" if "ema_model" in payload["state_dicts"] else "model"
    ema_sd = {k.replace("module.", ""): v
              for k, v in payload["state_dicts"][ema_key].items()
              if not k.endswith("position_ids")}
    _sd_result = inner.load_state_dict(ema_sd, strict=False)
    if _sd_result.missing_keys:
        print(f"    ⚠️  MISSING {len(_sd_result.missing_keys)} keys: {_sd_result.missing_keys[:3]}")
    else:
        print(f"    ✅ 0 missing keys — all weights loaded from checkpoint")
    if _sd_result.unexpected_keys:
        print(f"    ℹ️  {len(_sd_result.unexpected_keys)} unexpected keys (OK with strict=False)")
    print(f"    state dict loaded ({time.time()-t_start:.1f}s)")

    # ── [3] Move to GPU ──────────────────────────────────────────────────────
    print(f"[3] Moving model to {device}...")
    inner.to(device); inner.eval()
    if device.startswith("cuda"):
        allocated = torch.cuda.memory_allocated(device) / 1e9
        print(f"    model on GPU, VRAM used: {allocated:.2f} GB ({time.time()-t_start:.1f}s)")
    else:
        print(f"    model on {device} ({time.time()-t_start:.1f}s)")

    policy = inner
    cot_meta = {"cot_enabled": False}
    if not no_cot:
        cot_planner = create_planner(
            planner, model=llm_model, fallback_rule_based=not no_llm_fallback
        )
        policy = CoTOrchestratedPolicy(
            inner_policy=inner,
            planner=cot_planner,
            replan_every=replan_every,
            num_candidates=num_candidates,
            candidate_strategy=candidate_strategy,
            verbose=verbose_cot,
        )
        cot_meta = {
            "cot_enabled": True,
            "planner": planner,
            "replan_every": replan_every,
            "num_candidates": num_candidates,
            "candidate_strategy": candidate_strategy,
        }
        if planner == "llm":
            cot_meta["llm_model"] = llm_model

    # Normalizer sanity check
    try:
        _act_scale = inner.normalizer.params_dict["action"]["scale"]
        print(f"    normalizer.action.scale shape={tuple(_act_scale.shape)} min={_act_scale.min():.4f} max={_act_scale.max():.4f}")
    except Exception as _e:
        print(f"    ⚠️  normalizer check failed: {_e}")

    import gc
    print(f"[4] Collecting task hdf5 list...")
    hdf5_files = _get_libero_hdf5_files(cfg, task_subset)
    print(f"    {len(hdf5_files)} tasks found ({time.time()-t_start:.1f}s)")

    step_log = {}
    for i, hdf5_f in enumerate(hdf5_files):
        task_label = os.path.basename(hdf5_f)
        print(f"[4-{i+1}/{len(hdf5_files)}] Runner: {task_label}")
        t_r = time.time()
        er = hydra.utils.instantiate(cfg.task.env_runner, task_dir=hdf5_f, output_dir=output_dir)
        assert isinstance(er, BaseImageRunner)
        print(f"    runner ready ({time.time()-t_r:.1f}s), running eval...")
        step_log.update(er.run(policy))
        try: er.env.close()
        except Exception: pass
        del er
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"    done, RAM freed ({time.time()-t_start:.1f}s total)")

    test_scores = {k: v for k, v in step_log.items() if "test/" in k and "_mean_score" in k}
    step_log["test_mean_score"] = float(np.mean(list(test_scores.values()))) if test_scores else float("nan")

    json_log = {}
    for k, v in step_log.items():
        if isinstance(v, wandb.sdk.data_types.video.Video):
            json_log[k] = v._path
        else:
            try: json.dumps(v); json_log[k] = v
            except TypeError: json_log[k] = str(v)
    json_log.update(cot_meta)
    json_log["device"] = device
    tag = "baseline" if no_cot else planner
    out_path = os.path.join(output_dir, f"eval_cot_{tag}_{os.path.basename(checkpoint)}.json")
    with open(out_path, "w") as f:
        json.dump(json_log, f, indent=2, sort_keys=True)
    print("Saved", out_path)
    print(f"test_mean_score: {step_log['test_mean_score']}  (total {time.time()-t_start:.1f}s)")
    return step_log, json_log

# lr_scheduler.py 패치 — 구버전 import 스타일일 때만 (repo가 이미 패치됐으면 no-op)
from pathlib import Path
lr = Path("/content/unified_video_action/unified_video_action/model/common/lr_scheduler.py")
if lr.exists():
    text = lr.read_text()
    if "from diffusers.optimization import (\n    Union," in text:
        lr.write_text(text.replace(
            "from diffusers.optimization import (\n    Union,\n    SchedulerType,\n    Optional,\n    Optimizer,\n    TYPE_TO_SCHEDULER_FUNCTION,\n)",
            "from typing import Optional, Union\n\nfrom diffusers.optimization import (\n    SchedulerType,\n    Optimizer,\n    TYPE_TO_SCHEDULER_FUNCTION,\n)",
        ))
        print("patched lr_scheduler.py (old style)")
    else:
        print("lr_scheduler.py already patched")


/content/unified_video_action
lr_scheduler.py already patched


## 11. Smoke — 1 task × 1 ep (baseline / rule / LLM 비교)

In [17]:
SMOKE_TASK = ["moka_pot"]
SMOKE_KW = dict(n_test=1, n_train=0, n_envs=1, n_test_vis=0, max_steps=300, task_subset=SMOKE_TASK)

step_baseline, _ = run_libero10_cot_eval(
    output_dir="outputs/cot_smoke/baseline",
    no_cot=True,
    **SMOKE_KW,
)

step_rule, _ = run_libero10_cot_eval(
    output_dir="outputs/cot_smoke/rule",
    planner="rule",
    verbose_cot=True,
    **SMOKE_KW,
)

step_llm, _ = run_libero10_cot_eval(
    output_dir="outputs/cot_smoke/llm",
    planner="llm",
    llm_model="gpt-4o-mini",
    verbose_cot=True,
    **SMOKE_KW,
)

[1] Loading checkpoint: checkpoints/libero10.ckpt
    done (0.5s)
[2] Building policy (no workspace deepcopy / no optimizer)...
    [2a] Pre-caching CLIP from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


    CLIP cached (2.3s)
    [2b] Instantiating policy...
Working with z of shape (1, 16, 16, 16) = 4096 dimensions.
pretrained model not found:  /store/real/shuang/diffusion_policy_checkpoints/libero_10_image/unified-act-autoregressive-cant-proj-dataaug10-clip-2/checkpoints/best.ckpt
----------------------------------------------------------------------
task_modes ['video_model', 'dynamic_model', 'policy_model', 'inverse_model', 'full_dynamic_model']
----------------------------------------------------------------------
    policy built (6.9s)
    [2c] Loading ema_model state dict...
    state dict loaded (7.0s)
[3] Moving model to cuda:0...
    model on GPU, VRAM used: 3.74 GB (7.5s)
[4] Creating env runners (EGL/osmesa init happens here)...
Tasks (2):
  KITCHEN_SCENE3_turn_on_the_stove_and_put_the_moka_pot_on_it_demo.hdf5
  KITCHEN_SCENE8_put_both_moka_pots_on_the_stove_demo.hdf5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Created environment with name Libero_Kitchen_Tabletop_Manipulation
Action size is 7
ROBOMIMIC WARNING(
    No environment version found in dataset!
    Cannot verify if dataset and installed environment versions match
)
  runner 1/2 ready (2.0s)


...

Eval KITCHEN_SCENE8_put_both_moka_pots_on_the_stoveImage 1/1:  96%|█████████▌| 288/300 [03:02<00:07,  1.55it/s]

[CoT] replan -> KITCHEN SCENE8 put both moka pots on the stove | pick up the second moka pot
1. Perceive the location of the second moka pot. 2. Reason that it needs to be picked up. 3. Subgoal is to pick it up.
predict_action language_goal:  ['KITCHEN SCENE8 put both moka pots on the stove | pick up the second moka pot']
libero10 max_length 30


Eval KITCHEN_SCENE8_put_both_moka_pots_on_the_stoveImage 1/1:  99%|█████████▊| 296/300 [03:07<00:02,  1.53it/s]

[CoT] replan -> KITCHEN SCENE8 put both moka pots on the stove | pick up the second moka pot from the table
1. Perceive the location of the second moka pot. 2. Reason that it needs to be moved to the stove. 3. Subgoal: pick up the second moka pot from the table.
predict_action language_goal:  ['KITCHEN SCENE8 put both moka pots on the stove | pick up the second moka pot from the table']
libero10 max_length 30


Saved outputs/cot_smoke/llm/eval_cot_llm_libero10.ckpt.json
test_mean_score: 0.0  (total 398.3s)


## 12. Light — 10 tasks × 3 ep (LLM CoT, 논문 30-test 규모)

In [ ]:
step_light_llm, json_light_llm = run_libero10_cot_eval(
    output_dir="outputs/cot_light/llm",
    planner="llm",
    llm_model="gpt-4o-mini",
    n_test=3,
    n_envs=1,
    n_test_vis=0,
    verbose_cot=False,
)
print("LLM light mean:", step_light_llm.get("test_mean_score"))

## 13. 결과 비교표

In [18]:
import pandas as pd

rows = []
for name, log in [
    ("baseline", locals().get("step_baseline")),
    ("rule-CoT", locals().get("step_rule")),
    ("LLM-CoT smoke", locals().get("step_llm")),
    ("LLM-CoT light", locals().get("step_light_llm")),
]:
    if log is None:
        continue
    rows.append({"mode": name, "test_mean_score": log.get("test_mean_score", float("nan"))})

if rows:
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print("\n논문 Table I (UVA baseline, 50 ep): 0.90")
    print("논문 Suppl. VIII (30 ep): 0.93")
else:
    print("아직 평가 결과 없음 — 셀 10 이상 실행")

         mode  test_mean_score
     baseline              0.0
     rule-CoT              0.0
LLM-CoT smoke              0.0

논문 Table I (UVA baseline, 50 ep): 0.90
논문 Suppl. VIII (30 ep): 0.93


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 14. STUDY_SCENE1 재현성 확인 (n=5, baseline)

In [ ]:
step_a, _ = run_libero10_cot_eval(
    output_dir="outputs/debug_study",
    no_cot=True,
    n_test=5, n_train=0, n_envs=1, n_test_vis=0,
    task_subset=["STUDY_SCENE1"],
)
print("STUDY_SCENE1 baseline (n=5):", step_a.get("test_mean_score"))

## 트러블슈팅
- **`cot/` 없음** → 셀 3 fork URL이 LLM wrapper 브랜치인지 확인.
- **LLM이 rule로 fallback** → `OPENAI_API_KEY` 미설정. 셀 8 + trace에 `[LLM skipped]` 확인.
- **gym 설치 실패** → `colab_libero10_eval.ipynb`와 동일: 4a → 재시작 → 4b.
- **API 비용** → smoke는 1 task·1 ep; light는 replan마다 1회 API 호출 × (steps/8) × 30 ep.